## Mise en place

**Dans Colab, exécutez la cellule suivante une fois**, avant toutes les autres : elle dépose le module `mcdp_utils.py` et le dossier `data` du cours à côté du notebook.
Sur votre machine, où ces fichiers sont déjà en place, elle ne télécharge rien et ne change rien.

In [ ]:
"""Mise en place : le module et les donnees du cours, a cote du notebook."""
# Dans Colab, la machine pretee est vide : cette cellule telecharge depuis le
# depot du cours ce qui manque, et rien d'autre. Aucune installation.
import os
import urllib.request

RAW_BASE = "https://raw.githubusercontent.com/adilion1/cours-monte-carlo/main/notebooks/"
FICHIERS = ("mcdp_utils.py", "data/prices.csv", "data/returns_daily.csv")

os.makedirs("data", exist_ok=True)
_manquants = [_nom for _nom in FICHIERS if not os.path.exists(_nom)]
for _nom in _manquants:
    print("telechargement :", _nom)
    urllib.request.urlretrieve(RAW_BASE + _nom, _nom)

print("Prêt : module et données en place.")

# TP du chapitre 1 : cinq objets, aucune probabilité

**Master 2 MBFA · Optimisation dynamique et simulations de Monte-Carlo sous Python**
Chapitre 1, *Les objets de la finance qu'on peut observer*. Support : les cinq unités `u01` à `u05`.

---

## Ce que ce TP vous demande

Vous disposez de **4 151 séances de bourse** (5 janvier 2010 → 4 septembre 2026) et de six lignes
de portefeuille. Vous allez écrire **huit fonctions courtes** et vérifier, sur ces données, les cinq
chiffres du chapitre : ce qu'un placement de 1 000 € est devenu, ce que vaut « la » volatilité du
S&P 500, pourquoi un mélange de six lignes risquées est moins risqué que chacune d'elles, ce que
valent aujourd'hui 10 000 € promis dans un an, et à partir de quel niveau un call rapporte.

**Rien ici n'est simulé.** Le chapitre 1 ne contient pas un seul énoncé sur les chances que quelque
chose arrive : tout est mesuré sur des séances qui ont eu lieu, et tout est relançable. C'est le
chapitre 2 qui introduira le hasard.

**TP non noté.** Les trois TP notés du cours sont ceux des chapitres 4, 7 et 11. Ici, la note est
remplacée par les `verifier(...)` : chaque partie se corrige toute seule.

## Objectifs (vérifiables ici même)

| # | Vous saurez | Où c'est vérifié |
|---|---|---|
| **O-A** | passer d'une série de prix à ses rendements simples **et** à ses log-rendements, et dire lequel s'agrège dans le temps | partie A |
| **O-B** | annualiser un écart-type en nommant sa **fenêtre** et sa convention, et mesurer ce que vaut la règle en $\sqrt{h}$ | partie B |
| **O-C** | calculer la volatilité d'un mélange de deux actifs, et expliquer par les corrélations pourquoi le portefeuille passe sous sa ligne la plus prudente | partie C |
| **O-D** | actualiser un flux certain et pricer une obligation à trois flux, puis dire dans quel sens elle bouge quand le taux monte | partie D |
| **O-E** | remplir un tableau de payoff à huit cases, calculer un point mort avec la prime **capitalisée**, vérifier parité et bornes **sans modèle** | partie E |

## Temps

| Partie | Contenu | Durée |
|---|---|---|
| **0** | Trois prédictions écrites (obligatoire, non noté) | 3 min |
| **A** | Prix et rendements (u01) | 15 min |
| **B** | Volatilité et fenêtre (u02) | 20 min |
| **C** | Diversification (u03) | 15 min |
| **D** | Actualisation (u04) | 10 min |
| **E** | Payoffs, profit, parité (u05) | 12 min |
| | **Total mains sur clavier** | **75 min** |
| **Clôture** | Relecture des trois prédictions (à lire, pas à coder) | 5 min |
| | **Total du TP** | **80 min** |

## Avant la cellule 1 : les trois fichiers du cours

Si vous travaillez **en ligne**, téléversez d'abord dans la session Colab `mcdp_utils.py`, puis
`prices.csv` et `returns_daily.csv` dans un dossier `data/` : la marche à suivre, avec les clics,
est décrite une fois pour toutes dans `chapitre.md`, section « **Comment ouvrir le TP** ». Sans ces
**trois** fichiers, la cellule 1 s'arrête sur `ModuleNotFoundError: No module named 'mcdp_utils'`,
et les fichiers d'une session Colab disparaissent à chaque redémarrage. Si vous avez le dossier du
cours en local, il n'y a rien à faire : ouvrez ce notebook depuis `cours_v2/ch01/`.

## Trois conventions, à lire une fois

1. **Le notebook se lit et s'exécute strictement dans l'ordre**, du haut vers le bas. La cellule 1
   charge les données et définit les constantes ; toutes les autres en dépendent.
2. **Cellules à trous.** Vous **remplacez** le bloc `# TODO: … / raise NotImplementedError("TODO")`
   par votre code et **vous gardez tout ce qui suit** : le `return`, les vérifications et les
   affichages font partie de l'exercice. Tant que le `raise` est là, les lignes en dessous sont
   inatteignables : c'est normal, ce n'est pas une panne.
3. **`verifier(condition, message_ok, message_ko)`** affiche une phrase quand c'est juste, et sinon
   affiche ce qui était attendu **et où regarder**, avant de lever une `AssertionError`. Un échec
   n'est jamais un simple « faux ».

**Les cellules déjà écrites** emploient des fonctions numpy que vous n'avez pas à connaître
(`np.corrcoef`, `np.polyfit`, `np.concatenate`, `np.where`, `np.median`, …) : elles se **lisent**,
elles ne s'apprennent pas. Les fonctions que **vous** écrivez n'utilisent que les quinze primitives
de ch00 u06. Les cellules concernées portent la mention « cellule fournie ».

**Aucune cellule de ce TP ne tire un nombre au hasard.** La convention du cours (un `SEED`, un
`rng`, déclarés dans la cellule d'installation) est posée au ch00 u06 et servira pour la première
fois au chapitre 2 ; elle n'a rien à faire ici, et vous ne la verrez donc pas en cellule 1.

**Règles de calcul imposées** (`NOTATION.md`) : écart-type toujours en `ddof=1` ; annualisation en
**252** séances ; les fonctions fournies (`charger_fil_rouge`, `vol_portefeuille`, `payoff_call`,
`payoff_put`) sont **appelées, jamais réécrites** ; aucune boucle Python sur les 4 151 séances.

In [ ]:
"""Chapter 1 lab -- setup cell: imports, red-thread data, constants, sanity checks."""
%matplotlib inline

import sys

sys.path.insert(0, "..")  # cours_v2/ch01/ -> cours_v2/, where mcdp_utils.py lives

import matplotlib.pyplot as plt
import numpy as np

from mcdp_utils import (charger_fil_rouge, payoff_call, payoff_put, set_style,
                        verifier, vol_portefeuille)

set_style()   # same figure style as ch00; no random number is drawn anywhere in ch01

DONNEES = charger_fil_rouge()
DATES = DONNEES["dates"]                 # 4151 trading days, datetime64[D]
PRIX = DONNEES["prix"]                   # asset -> ndarray of closing prices
RLOG = DONNEES["rendements_log"]         # asset -> ndarray of daily log returns
POIDS = DONNEES["poids"]                 # asset -> weight, sums to 1

LIGNES = ["SP500", "CAC40", "AAPL", "LVMH", "GLD", "TLT"]
S0 = DONNEES["S0"]                       # 7718.60, SP500 close on 2026-09-04
K = S0                                   # at-the-money strike
r = DONNEES["r"]                         # 0.0375, DTB3 of 2026-09-03, continuous
T = 1.0                                  # maturity, in years
PRIME_CALL, PRIME_PUT = 546.2794, 262.1918   # course reference premiums (ch01 u05)


def rappel(nom: str, gabarit: str = "{}", defaut: str = "(à compléter)") -> str:
    """Format a value bound by an earlier cell, or a placeholder if it is missing.

    Used only by the closing cell, so that the statement notebook can be run top to
    bottom before every TODO has been filled in.
    """
    valeur = globals().get(nom, None)
    return defaut if valeur is None else gabarit.format(valeur)


print(f"python {sys.version.split()[0]} | numpy {np.__version__}")
print(f"{DATES.size} seances, du {DATES[0]} au {DATES[-1]} | {len(PRIX)} actifs")
print(f"S0 = {S0:,.2f} | r = {r:.4f} | T = {T} | poids = {POIDS}")

assert DATES.size == 4151, "the red-thread panel should hold 4151 aligned sessions"
assert abs(S0 - 7718.60) < 1e-6, "S0 moved: check data/stats_summary.md"
assert abs(sum(POIDS.values()) - 1.0) < 1e-12, "portfolio weights must sum to 1"

---

## Partie 0 : Trois prédictions écrites (3 min, obligatoire, non noté)

**Ne lancez aucune autre cellule avant d'avoir rempli celle-ci.** Une prédiction écrite avant le
calcul ne sert pas à avoir raison : elle sert à rendre visible l'écart entre ce que vous croyiez et
ce que les données disent. Sans elle, on lit le bon chiffre et on se persuade qu'on l'attendait.

Trois questions, trois nombres, aucun calcul :

1. **`capital_2010_euros`** : 1 000 € placés sur le S&P 500 le 5 janvier 2010, jamais touchés.
   Combien y a-t-il le 4 septembre 2026 ?
2. **`vol_portefeuille_pct`** : les six lignes du portefeuille ont des volatilités annualisées
   comprises entre 14,98 % (TLT) et 28,27 % (AAPL). Quelle est celle du mélange 40/20/10/10/10/10,
   **en pour-cent** ?
3. **`point_mort_call`** : vous payez **546,28** aujourd'hui le droit d'acheter une unité de S&P 500
   à **7 718,60** dans un an. À partir de quel niveau du S&P 500 votre opération est-elle gagnante ?

Écrivez vos trois nombres. La clôture les relira à côté des chiffres mesurés.

In [ ]:
def mes_predictions() -> dict[str, float]:
    """The three numbers written down BEFORE running anything else.

    Returns
    -------
    dict
        Keys ``capital_2010_euros`` (euros), ``vol_portefeuille_pct`` (annualised
        volatility of the six-line portfolio, in percent) and ``point_mort_call``
        (the S&P 500 level above which the call trade makes money, in index points).
    """
    # TODO: Renvoyer vos trois predictions dans un dictionnaire portant les trois
    #      cles de la docstring ; n'importe quel nombre honnete convient, rien n'est
    #      note ici.
    raise NotImplementedError("TODO")


PREDICTIONS = mes_predictions()

assert set(PREDICTIONS) == {
    "capital_2010_euros",
    "vol_portefeuille_pct",
    "point_mort_call",
}, "the dict must carry exactly the three keys of the docstring"
assert PREDICTIONS["capital_2010_euros"] > 0.0
assert 0.0 < PREDICTIONS["vol_portefeuille_pct"] < 100.0
assert PREDICTIONS["point_mort_call"] > 0.0
for cle, valeur in PREDICTIONS.items():
    print(f"{cle:>22s} : {valeur:,.2f}")

---

## Partie A : Un prix n'est pas un rendement (15 min, u01)

**Le problème.** Un épargnant vous montre deux relevés. Sur le premier, le S&P 500 est passé de
1 136,52 à 7 718,60 ; sur le second, un fonds qui détient exactement les mêmes sociétés affiche une
performance nettement supérieure. Il vous demande lequel des deux ment. Aucun des deux : le premier
suit un **indice de prix**, le second un **indice dividendes réinvestis**, et l'écart est le
dividende. Encore faut-il savoir passer d'une série de prix à un rendement, dans les deux échelles.

Vous écrivez les deux conversions du chapitre :

| Fonction | Ce qu'elle rend | Formule |
|---|---|---|
| `rendements_simples(prix)` | $R_t = S_t/S_{t-1} - 1$, **une case de moins** que `prix` | l'échelle qui se **pondère entre actifs** |
| `log_rendements(prix)` | $\ell_t = \ln(S_t/S_{t-1})$, **une case de moins** elle aussi | l'échelle qui s'**additionne dans le temps** |

Les deux doivent être **vectorisées** : une seule expression numpy sur des tranches, aucune boucle
`for` sur les 4 151 séances.

In [ ]:
def rendements_simples(prix: np.ndarray) -> np.ndarray:
    """Simple returns of a price series: R_t = S_t / S_{t-1} - 1.

    Parameters
    ----------
    prix : ndarray, shape (n,)
        Closing prices, in chronological order, all strictly positive.

    Returns
    -------
    ndarray, shape (n - 1,)
        Simple returns, in decimal.  Element ``k`` is the return **from**
        ``prix[k]`` **to** ``prix[k + 1]``: the array is one cell shorter than the
        prices, because it takes two prices to make one return.
    """
    # TODO: Renvoyer les rendements simples de facon vectorisee, avec les deux
    #      tranches prix[1:] et prix[:-1], et sans aucune boucle Python.
    raise NotImplementedError("TODO")


def log_rendements(prix: np.ndarray) -> np.ndarray:
    """Log returns of a price series: ell_t = log(S_t / S_{t-1}).

    Parameters
    ----------
    prix : ndarray, shape (n,)
        Closing prices, in chronological order, all strictly positive.

    Returns
    -------
    ndarray, shape (n - 1,)
        Log returns, in decimal.  Their **sum** over any period is the log return of
        the whole period, which is exactly why the course measures volatility on them.
    """
    # TODO: Renvoyer les log-rendements de facon vectorisee, avec np.log applique au
    #      rapport des deux memes tranches.
    raise NotImplementedError("TODO")

In [ ]:
"""Check the two conversions on a three-price series everyone can redo by hand."""
essai = np.array([100.0, 150.0, 75.0])       # +50 %, then -50 %
simples, logs = rendements_simples(essai), log_rendements(essai)

verifier(
    abs(simples[1] + 0.5) < 1e-12,
    f"OK : passer de 150 a 75 est bien -50 % (obtenu {simples[1]:.4f}).",
    "attendu -0,5 en deuxieme case : la division doit etre prix[1:] / prix[:-1] - 1, "
    "et non prix[:-1] / prix[1:] - 1 (relire la docstring de rendements_simples).",
)
verifier(
    abs(logs.sum() + 0.2876820724) < 1e-9,
    f"OK : les deux log-rendements s'additionnent a {logs.sum():.10f}, "
    f"soit une richesse finale de {np.exp(logs.sum()):.4f} fois la mise (-25 %).",
    "attendu -0,2876820724 pour la somme : +50 % puis -50 % donne ln(1,5) + ln(0,5). "
    "Si vous obtenez 0, vous avez additionne les rendements simples (u01 section 4).",
)
verifier(
    simples.size == essai.size - 1 and logs.size == essai.size - 1,
    "OK : les deux tableaux ont bien une case de moins que celui des prix.",
    "attendu deux tableaux de longueur n - 1 : il faut deux prix pour faire un "
    "rendement, donc 3 prix donnent 2 rendements (u01, verifiez-vous n. 5).",
)
print(f"simples = {np.round(simples, 4)}   logs = {np.round(logs, 6)}")

In [ ]:
"""The saver's question, on the two indices of the panel."""
base100 = {a: 100.0 * PRIX[a] / PRIX[a][0] for a in ("SP500", "SPY")}
capital = {a: 10.0 * base100[a][-1] for a in base100}   # 1000 EUR invested on 2010-01-05

for actif in ("SP500", "SPY"):
    somme_logs = log_rendements(PRIX[actif]).sum()
    print(f"{actif:>6s} : base 100 -> {base100[actif][-1]:7.1f} | {capital[actif]:9,.2f} EUR "
          f"| somme des log-rendements {somme_logs:.4f} -> {100 * (np.exp(somme_logs) - 1):.2f} %")

verifier(
    abs(base100["SP500"][-1] - 679.1) < 0.2 and abs(base100["SPY"][-1] - 908.2) < 0.2,
    f"OK : 679,1 contre 908,2 en base 100, soit {base100['SPY'][-1] - base100['SP500'][-1]:.1f} "
    f"points d'ecart, et {capital['SPY'] - capital['SP500']:.0f} EUR sur une mise de 1 000 EUR.",
    "attendu 679,1 (SP500) et 908,2 (SPY) : la base 100 se calcule en divisant toute la "
    "serie par sa PREMIERE valeur, PRIX[actif][0], pas par la derniere.",
)
verifier(
    abs(np.exp(log_rendements(PRIX["SP500"]).sum()) - PRIX["SP500"][-1] / PRIX["SP500"][0]) < 1e-9,
    "OK : la somme des 4 150 log-rendements, repassee en exponentielle, redonne "
    "exactement le rapport des prix extremes. C'est la propriete d'additivite.",
    "attendu exp(somme des log-rendements) = dernier prix / premier prix. Si l'ecart "
    "n'est pas nul, log_rendements n'utilise pas le rapport de deux prix CONSECUTIFS.",
)

### Le piège de la partie A : agréger six lignes avec la mauvaise échelle

Le 12 mars 2020, les six lignes du portefeuille ont toutes bougé le même jour. La question est de
savoir ce qu'a fait **le portefeuille**. Deux calculs sont possibles : pondérer les rendements
**simples** des six lignes, ou pondérer leurs **log-rendements**. L'un est le bon, l'autre est faux,
et la cellule suivante mesure ce que coûte l'erreur en une seule séance.

Une remarque d'indexation, utile ici et partout ensuite : `rendements_simples(prix)` a **une case de
moins** que `prix`, donc le rendement de la séance `DATES[i]` se lit en position `i - 1`.

In [ ]:
"""How much does aggregating six assets on the wrong scale cost, in one session?"""
i = int(np.where(DATES == np.datetime64("2020-03-12"))[0][0])
simples_du_jour = {a: rendements_simples(PRIX[a])[i - 1] for a in LIGNES}
logs_du_jour = {a: RLOG[a][i] for a in LIGNES}

ptf_simple = sum(POIDS[a] * simples_du_jour[a] for a in LIGNES)     # the correct one
ptf_log = sum(POIDS[a] * logs_du_jour[a] for a in LIGNES)           # the mistake

print(f"seance du {DATES[i]}")
for a in LIGNES:
    print(f"  {a:>6s} : simple {100 * simples_du_jour[a]:7.2f} %   log {100 * logs_du_jour[a]:7.2f} %")
print(f"portefeuille, rendements simples ponderes : {100 * ptf_simple:.4f} %")
print(f"portefeuille, log-rendements ponderes     : {100 * ptf_log:.4f} %  <- faux")

verifier(
    abs(1e4 * (ptf_simple - ptf_log) - 45.8) < 0.5,
    f"OK : {1e4 * (ptf_simple - ptf_log):.1f} points de base d'ecart en UNE seance. Entre "
    "actifs on pondere des rendements simples ; les log ne s'additionnent que dans le temps.",
    "attendu environ 45,8 points de base d'ecart (u01 section 5). Verifiez que "
    "simples_du_jour lit bien la position i - 1 du tableau des rendements.",
)
verifier(
    max(abs(np.exp(logs_du_jour[a]) - 1 - simples_du_jour[a]) for a in LIGNES) < 1e-12,
    "OK : ligne par ligne, R = exp(ell) - 1 tombe au douzieme chiffre. Les deux echelles "
    "decrivent le MEME mouvement ; c'est l'agregation qui les separe, pas la mesure.",
    "attendu R = exp(ell) - 1 sur chaque ligne : si l'ecart n'est pas nul, l'indice i - 1 "
    "ne designe pas la meme seance que RLOG[a][i].",
)

In [ ]:
"""Figure A -- price index against total-return index, base 100, log scale."""
# Cellule FOURNIE : lisez la figure, pas le code.
fig, ax = plt.subplots(figsize=(8.0, 4.4))
for actif, couleur, etiquette in (
    ("SP500", "#1f4e79", "SP500 (indice de prix)"),
    ("SPY", "#2e7d32", "SPY (dividendes reinvestis)"),
):
    ax.plot(DATES, 100.0 * PRIX[actif] / PRIX[actif][0], lw=1.2, color=couleur, label=etiquette)
ax.set_yscale("log")
ax.set_yticks([100, 200, 400, 800])
ax.set_yticklabels(["100", "200", "400", "800"])
fin = {a: 100.0 * PRIX[a][-1] / PRIX[a][0] for a in ("SP500", "SPY")}
ax.annotate("", xy=(DATES[-1], fin["SP500"]), xytext=(DATES[-1], fin["SPY"]),
            arrowprops=dict(arrowstyle="<->", color="#2e7d32", lw=1.4))
ax.text(DATES[2350], 108,
        f"{fin['SPY'] - fin['SP500']:.1f} points d'ecart en base 100 : les dividendes",
        fontsize=9, color="#2e7d32")
ax.set_ylabel("base 100 au 2010-01-05 (echelle log)")
ax.set_title("Meme marche, meme forme, deux niveaux : la difference est le dividende")
ax.legend(loc="upper left", fontsize=9)
plt.show()

---

## Partie B : La volatilité et sa fenêtre (20 min, u02)

**Le problème.** Un client vous demande « la volatilité du S&P 500 ». Vous avez 4 151 séances sous
la main. Il n'existe pas *une* réponse : selon la fenêtre retenue, le même indice affiche de 6,7 %
à 34,8 %. Cette partie construit le chiffre, puis mesure de combien il bouge, et ce que vaut la
règle qui prétend passer d'un horizon à un autre.

Vous écrivez une fonction, et une seule :

```
volatilite_annualisee(log_rend, periodes=252) -> float
```

Écart-type des log-rendements en `ddof=1`, multiplié par $\sqrt{\text{periodes}}$. Le paramètre
`periodes` n'est pas une décoration : c'est lui qui porte la convention 252 contre 365, et le piège
de cette partie.

In [ ]:
def volatilite_annualisee(log_rend: np.ndarray, periodes: int = 252) -> float:
    """Annualised volatility of a series of log returns.

    Parameters
    ----------
    log_rend : ndarray, shape (n,)
        Daily log returns, in decimal.
    periodes : int, optional
        Number of periods per year used for the annualisation; 252 trading days by
        default.  Passing 365 on daily *trading* data is the classic mistake, measured
        two cells below.

    Returns
    -------
    float
        Empirical standard deviation with ``ddof=1``, times ``sqrt(periodes)``.
    """
    # TODO: Renvoyer l'ecart-type de log_rend calcule avec ddof=1, multiplie par la
    #      racine carree de periodes, converti en float.
    raise NotImplementedError("TODO")

In [ ]:
"""One asset, three windows -- and the same series annualised the other way."""
ell = RLOG["SP500"]
sigma_1an = volatilite_annualisee(ell[-252:])
sigma_5ans = volatilite_annualisee(ell[-1260:])
sigma_totale = volatilite_annualisee(ell)
sigma_365 = volatilite_annualisee(ell[-252:], periodes=365)

print(f"252 dernieres seances   : {100 * sigma_1an:6.2f} %")
print(f"1 260 dernieres (5 ans) : {100 * sigma_5ans:6.2f} %")
print(f"les 4 151 seances       : {100 * sigma_totale:6.2f} %")
print(f"252 dernieres, en V365  : {100 * sigma_365:6.2f} %  <- meme serie, autre convention")
print(f"                          (valeur exacte {100 * sigma_365:.4f} % ; u02 affiche 15,45 %,")
print("                           parce qu'il repart du 12,84 % deja arrondi)")

verifier(
    abs(sigma_1an - 0.1284) < 1e-4 and abs(sigma_5ans - 0.1696) < 1e-4,
    f"OK : {100 * sigma_1an:.2f} % sur un an contre {100 * sigma_5ans:.2f} % sur cinq ans. "
    "Quatre points d'ecart sur le meme actif : la fenetre fait partie du chiffre.",
    "attendu 0,1284 et 0,1696 (cf. data/stats_summary.md section 3). Si vous obtenez des "
    "valeurs proches de 0,008, la multiplication par sqrt(252) manque ; si vous obtenez "
    "0,12817 et 0,16949, c'est ddof=1 qui manque (la cellule suivante le teste seule).",
)
verifier(
    abs(volatilite_annualisee(np.array([0.01, -0.01, 0.02]), periodes=1) - 0.015275) < 1e-6,
    "OK : sur trois valeurs, ddof=1 donne 0,015275. Ce controle-la est fait pour tomber "
    "si ddof manque : sur le fil rouge l'ecart serait de 0,2 %, invisible a l'oeil.",
    "attendu 0,015275 sur [0.01, -0.01, 0.02] : avec ddof=0 on obtient 0,012472, soit 22 % "
    "de moins. np.std(..., ddof=1) divise la somme des carres par n - 1, pas par n.",
)
verifier(
    abs(sigma_365 / sigma_1an - np.sqrt(365 / 252)) < 1e-9,
    f"OK : passer de 252 a 365 multiplie la volatilite par {sigma_365 / sigma_1an:.4f}, soit "
    f"+{100 * (sigma_365 / sigma_1an - 1):.1f} % en relatif, pour un caractere change.",
    "attendu un rapport egal a sqrt(365/252) : le parametre periodes doit entrer sous la "
    "racine, et nulle part ailleurs.",
)

> **Piège classique du chapitre : changer de convention en route.** La cellule ci-dessus vient de
> l'afficher : **12,84 %** en $\sqrt{252}$, **15,46 %** en $\sqrt{365}$ (15,4555 % exactement ; u02
> affiche 15,45 %, parce qu'il refait le calcul depuis le 12,84 % déjà arrondi), sur les mêmes
> nombres. Aucune des deux n'est fausse en soi. La faute est de mélanger les deux dans un même
> calcul, parce qu'elles ne comptent pas la même chose : la **volatilité** s'annualise sur le
> **nombre d'observations** (un week-end ne produit aucun rendement observable), tandis qu'une
> **durée de contrat** se compte en **années calendaires**. Retenez l'ordre de grandeur : un
> caractère changé déplace la volatilité d'un cinquième.

### Prédiction, avant la mesure

$\hat\sigma_{\text{jour}} = 1{,}090\,\%$ sur toute la période. La règle du cours annonce que
l'écart-type d'un rendement cumulé sur $h$ séances vaut $\hat\sigma_{\text{jour}}\sqrt{h}$.

**Écrivez votre prédiction pour $h = 21$** (un mois de bourse), en pour-cent, avant de lancer la
mesure. La cellule d'après la confrontera à l'écart-type réellement observé sur toutes les fenêtres
de 21 séances de nos données.

In [ ]:
def prediction_21_jours() -> float:
    """Your predicted standard deviation of the 21-session cumulated return, in percent.

    Returns
    -------
    float
        A number in percent, e.g. ``4.5`` for 4,5 %.  Written before the measurement.
    """
    # TODO: Renvoyer, en pourcentage, l'ecart-type attendu sur 21 seances quand une
    #      seance vaut 1,090 % : la regle du cours donne 1.090 * sqrt(21).
    raise NotImplementedError("TODO")


print(f"prediction pour h = 21 : {prediction_21_jours():.3f} %")

In [ ]:
"""Measured standard deviation of h-session returns against the square-root rule."""
# Cellule FOURNIE : lisez le resultat, pas le code. Rien a completer ici, et ni cumsum
# ni polyfit ne sont a retenir (u02 section 6 explique ce que la pente mesure).
cumul = np.cumsum(ell)
horizons = np.array([1, 2, 5, 10, 21, 42, 63])
mesure = np.empty(horizons.size)
for k, h in enumerate(horizons):
    # Overlapping windows: sum of h consecutive log returns, for every possible start.
    agrege = cumul[h - 1:] - np.concatenate(([0.0], cumul[:-h]))
    mesure[k] = agrege.std(ddof=1)
regle = np.std(ell, ddof=1) * np.sqrt(horizons)

print("  h |  mesure |  regle  | rapport")
for k, h in enumerate(horizons):
    print(f"{h:3d} | {100 * mesure[k]:6.3f} % | {100 * regle[k]:6.3f} % | {mesure[k] / regle[k]:.2f}")
pente = float(np.polyfit(np.log(horizons), np.log(mesure), 1)[0])
print(f"pente log-log mesuree : {pente:.2f}  (la regle en sqrt(h) impose 0,50)")

verifier(
    abs(mesure[horizons == 21][0] - 0.04368) < 5e-4,
    f"OK : {100 * mesure[horizons == 21][0]:.3f} % mesures sur 21 seances, contre "
    f"{100 * regle[horizons == 21][0]:.3f} % annonces par la regle. La regle est haute de 14 %.",
    "attendu 4,368 % pour h = 21 : les fenetres doivent etre CHEVAUCHANTES (une par date de "
    "depart possible), et l'ecart-type se prend sur la somme des h log-rendements.",
)
verifier(
    0.40 < pente < 0.48,
    f"OK : pente mesuree {pente:.2f} au lieu de 0,50. La racine de t est une convention de "
    "calcul, bonne a un mois pres sur ces donnees, pas une propriete des marches.",
    "attendu une pente comprise entre 0,40 et 0,48 : elle se lit par np.polyfit sur les "
    "logarithmes des horizons et des ecarts-types mesures.",
)

In [ ]:
"""Same mean, three dispersions: which fund ends with the most money?"""
fonds = {
    "Alpha": np.array([-8.0, 5.0, 6.0, 7.0, 8.0, 18.0]) / 100.0,
    "Beta": np.array([-8.0, 0.0, 4.0, 10.0, 12.0, 18.0]) / 100.0,
    "Gamma": np.array([-8.0, -6.0, 4.0, 12.0, 16.0, 18.0]) / 100.0,
}
capitaux = {nom: 100.0 * np.prod(1.0 + rendements) for nom, rendements in fonds.items()}
for nom, rendements in fonds.items():
    print(f"{nom:>6s} : moyenne {100 * rendements.mean():.2f} % | ecart-type "
          f"{100 * rendements.std(ddof=1):5.2f} | 100 EUR deviennent {capitaux[nom]:.2f} EUR")

verifier(
    abs(capitaux["Alpha"] - 139.63) < 0.01 and capitaux["Alpha"] > capitaux["Gamma"],
    f"OK : {capitaux['Alpha']:.2f} EUR contre {capitaux['Gamma']:.2f} EUR pour la meme moyenne "
    "de 6 %. Ce qu'on encaisse est la moyenne geometrique, plus petite d'environ un demi-carre "
    "de volatilite par an.",
    "attendu 139,63 EUR pour Alpha : les six rendements se COMPOSENT (np.prod de 1 + R), ils "
    "ne s'additionnent pas (u01 section 4).",
)
ecart_annuel = 0.5 * (fonds["Gamma"].std(ddof=1) ** 2 - fonds["Alpha"].std(ddof=1) ** 2)
print(f"un demi-carre d'ecart de volatilite : {100 * ecart_annuel:.2f} point par an, "
      f"soit {100 * (capitaux['Alpha'] / capitaux['Gamma'] - 1):.2f} % de capital sur six ans")

In [ ]:
"""Figure B -- the volatility measured on every rolling window of 252 sessions."""
# Cellule FOURNIE : lisez la figure et les trois chiffres imprimes, pas le code.
fenetre = 252
n = ell.size
cum1, cum2 = np.cumsum(ell), np.cumsum(ell * ell)
fin = np.arange(fenetre, n + 1)                      # index just past each window
somme = cum1[fin - 1] - np.concatenate(([0.0], cum1[: n - fenetre]))
somme2 = cum2[fin - 1] - np.concatenate(([0.0], cum2[: n - fenetre]))
variance = (somme2 - somme * somme / fenetre) / (fenetre - 1)
glissante = 100.0 * np.sqrt(variance * 252)
dates_fin = DATES[fin - 1]

fig, ax = plt.subplots(figsize=(8.6, 4.4))
ax.plot(dates_fin, glissante, lw=1.0, color="#1f4e79")
for niveau, texte, couleur in (
    (100 * np.std(ell[-252:], ddof=1) * np.sqrt(252), "12,84 % : les 252 dernieres seances", "#c00000"),
    (100 * np.std(ell[-1260:], ddof=1) * np.sqrt(252), "16,96 % : les 1 260 dernieres", "#2e7d32"),
    (float(np.median(glissante[::5])), "15,00 % : mediane des 780 fenetres", "#6a3d9a"),
):
    ax.axhline(niveau, ls="--", lw=1.2, color=couleur, label=texte)
ax.set_ylabel("volatilite annualisee (%)")
ax.set_xlabel("date de fin de la fenetre de 252 seances")
ax.set_title("Un seul actif, un seul calcul, de 6,7 % a 34,8 % selon la fenetre")
ax.legend(loc="upper right", fontsize=8.5)
plt.show()

# The 780 windows of u02 are spaced five sessions apart; the curve above draws all 3 900.
pas5, dates5 = glissante[::5], dates_fin[::5]
print(f"{pas5.size} fenetres d'un an, espacees de 5 seances :")
print(f"  minimum {pas5.min():.2f} % (fenetre finissant le {dates5[pas5.argmin()]})")
print(f"  maximum {pas5.max():.2f} % (fenetre finissant le {dates5[pas5.argmax()]})")
print(f"  mediane {np.median(pas5):.2f} % : un facteur cinq sous le meme nom")

---

## Partie C : Diversifier n'est pas moyenner (15 min, u03)

**Le problème.** Vous devez présenter le risque du portefeuille du cours à un comité. Les six lignes
affichent 17,31 %, 19,61 %, 28,27 %, 28,00 %, 16,85 % et 14,98 % de volatilité annualisée : aucune
sous 14,98 %. Quelqu'un propose d'annoncer la moyenne pondérée, 19,65 %. C'est faux, et la partie C
mesure de combien.

Vous écrivez la formule à **deux** actifs ; au-delà, la fonction `vol_portefeuille()` vous est
fournie et sera **appelée**, pas réécrite :

$$\sigma_{\text{ptf}}^2 = w_1^2\sigma_1^2 + w_2^2\sigma_2^2 + 2\,w_1 w_2\,\rho\,\sigma_1\sigma_2$$

In [ ]:
def vol_deux_actifs(sigma1: float, sigma2: float, rho: float, w1: float) -> float:
    """Annualised volatility of a two-asset portfolio.

    Parameters
    ----------
    sigma1, sigma2 : float
        Annualised volatilities of the two assets, in decimal.
    rho : float
        Their correlation, between -1 and +1.
    w1 : float
        Weight of the first asset; the second gets ``1 - w1``.

    Returns
    -------
    float
        ``sqrt(w1^2 sigma1^2 + w2^2 sigma2^2 + 2 w1 w2 rho sigma1 sigma2)``.  It is the
        variances that add up, never the standard deviations -- hence the square root.
    """
    w2 = 1.0 - w1
    # TODO: Renvoyer la racine carree de la variance a trois termes ecrite dans la
    #      docstring, avec w2 = 1 - w1.
    raise NotImplementedError("TODO")

In [ ]:
"""Three 50/50 mixes, and the correlations that make them work."""
vols = {a: volatilite_annualisee(RLOG[a]) for a in LIGNES}
rho = {(x, y): float(np.corrcoef(RLOG[x], RLOG[y])[0, 1])
       for x, y in (("SP500", "TLT"), ("SP500", "GLD"), ("SP500", "AAPL"))}

# Le controle passe AVANT l'affichage : un tableau de chiffres faux imprime en premier
# reste dans la tete du lecteur, meme quand l'assert le contredit la ligne suivante.
verifier(
    abs(vol_deux_actifs(0.1731, 0.1498, -0.290, 0.5) - 0.0966) < 5e-4,
    f"OK : {100 * vol_deux_actifs(0.1731, 0.1498, -0.290, 0.5):.2f} % pour moitie SP500 moitie "
    "TLT, contre 16,14 % en moyennant les deux volatilites. Le terme croise est negatif.",
    "attendu 0,0966 : le troisieme terme vaut 2*w1*w2*rho*sigma1*sigma2 et porte le SIGNE de "
    "rho ; la racine se prend sur la somme entiere, pas terme a terme.",
)
verifier(
    abs(rho[("SP500", "TLT")] + 0.290) < 5e-3,
    f"OK : rho(SP500, TLT) = {rho[('SP500', 'TLT')]:+.3f}, la seule paire franchement negative "
    "du panel. Les obligations longues montent quand les actions baissent.",
    "attendu environ -0,290 (data/stats_summary.md section 2) : np.corrcoef rend une matrice "
    "2 x 2, la correlation cherchee est en position [0, 1].",
)
verifier(
    abs(vol_deux_actifs(0.20, 0.20, 1.0, 0.5) - 0.20) < 1e-12,
    "OK : a rho = +1 le melange vaut exactement la moyenne ponderee, donc le gain de "
    "diversification est NUL. Ce n'est pas le nombre de lignes qui fait la diversification.",
    "attendu exactement 0,20 : a rho = 1 la formule doit se reduire a (w1*sigma1 + w2*sigma2)^2 "
    "sous la racine.",
)

print("paire         |   rho  | melange 50/50 | moyenne des deux")
for (x, y), valeur in rho.items():
    melange = vol_deux_actifs(vols[x], vols[y], valeur, 0.5)
    print(f"{x:>5s} / {y:<5s} | {valeur:+.3f} |    {100 * melange:5.2f} %    |     {50 * (vols[x] + vols[y]):5.2f} %")

In [ ]:
"""The six-line portfolio: what the provided function says, against the weighted average."""
vols = {a: volatilite_annualisee(RLOG[a]) for a in LIGNES}
moyenne_ponderee = sum(POIDS[a] * vols[a] for a in LIGNES)
sigma_ptf = vol_portefeuille(RLOG, POIDS)          # provided function: called, not rewritten

for a in LIGNES:
    print(f"  {a:>6s} : {100 * vols[a]:5.2f} %  (poids {100 * POIDS[a]:.0f} %)")
print(f"moyenne ponderee des six : {100 * moyenne_ponderee:.2f} %")
print(f"volatilite du melange    : {100 * sigma_ptf:.2f} %")
print(f"gain de diversification  : {100 * (moyenne_ponderee - sigma_ptf):.2f} points")
print()

verifier(
    abs(sigma_ptf - 0.13417) < 1e-4 and abs(moyenne_ponderee - 0.19655) < 1e-4,
    f"OK : {100 * sigma_ptf:.2f} % contre {100 * moyenne_ponderee:.2f} %. Le melange est sous "
    f"sa ligne la plus prudente ({100 * min(vols.values()):.2f} % pour TLT) : c'est le resultat "
    "de l'unite u03.",
    "attendu 0,13417 pour le melange et 0,19655 pour la moyenne ponderee. vol_portefeuille "
    "prend les DEUX dictionnaires (rendements puis poids) ; la moyenne ponderee se calcule sur "
    "les volatilites annualisees, pas sur les rendements.",
)
verifier(
    sigma_ptf < min(vols.values()),
    "OK : la volatilite du portefeuille passe sous celle de chacune de ses six lignes.",
    "attendu un melange strictement sous le minimum des six volatilites : si ce n'est pas le "
    "cas, verifiez que vols est calcule ligne par ligne sur RLOG et non sur les prix.",
)
print("Note de convention : data/stats_summary.md publie 13,40 % pour ce meme portefeuille,")
print("calcule sur les rendements SIMPLES ponderes, quand vol_portefeuille passe par les")
print("log-rendements et rend 13,42 %. Aucun des deux n'est faux : l'ecart est celui de u01,")
print("et le cours n'ecrit jamais l'un sans dire lequel c'est.")

In [ ]:
"""Figure C -- the six volatilities, the mix, and the correlation map."""
# Cellule FOURNIE : lisez la figure, pas le code.
# Recomputed here with numpy so that the figure is drawn even before the two
# functions of this part are written.
vols = {a: float(np.std(RLOG[a], ddof=1) * np.sqrt(252)) for a in LIGNES}
moyenne_ponderee = sum(POIDS[a] * vols[a] for a in LIGNES)
sigma_ptf = vol_portefeuille(RLOG, POIDS)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.4, 4.4))

hauteurs = [100 * vols[a] for a in LIGNES] + [100 * moyenne_ponderee, 100 * sigma_ptf]
etiquettes = LIGNES + ["moyenne ponderee", "PORTEFEUILLE"]
couleurs = ["#7f9fc0"] * 6 + ["#c00000", "#2e7d32"]
ax1.bar(etiquettes, hauteurs, color=couleurs)
ax1.axhline(100 * min(vols.values()), color="grey", ls=":", lw=1.2)
ax1.set_ylim(0, 34)
ax1.set_xlim(-0.7, 8.9)
ax1.text(-0.6, 31.4, "ligne pointillee : 14,98 %, la plus prudente des six lignes",
         fontsize=8.5, color="grey")
ax1.annotate("", xy=(7, 100 * sigma_ptf), xytext=(7, 100 * moyenne_ponderee),
             arrowprops=dict(arrowstyle="<->", color="black", lw=1.4))
ax1.text(7.5, 16.0, f"{100 * (moyenne_ponderee - sigma_ptf):.2f}\npoints", fontsize=9)
ax1.set_ylabel("volatilite annualisee (%)")
ax1.set_title("(a) Le melange passe sous chacune de ses lignes")
ax1.tick_params(axis="x", labelsize=8, rotation=30)

matrice = np.array([[np.corrcoef(RLOG[x], RLOG[y])[0, 1] for y in LIGNES] for x in LIGNES])
image = ax2.imshow(matrice, cmap="RdBu_r", vmin=-1, vmax=1)
ax2.set_xticks(range(6))
ax2.set_xticklabels(LIGNES, fontsize=8, rotation=45)
ax2.set_yticks(range(6))
ax2.set_yticklabels(LIGNES, fontsize=8)
for x in range(6):
    for y in range(6):
        ax2.text(y, x, f"{matrice[x, y]:+.2f}", ha="center", va="center", fontsize=8,
                 color="white" if abs(matrice[x, y]) > 0.6 else "black")
fig.colorbar(image, ax=ax2, fraction=0.046)
ax2.set_title("(b) Qui bouge avec qui : la colonne TLT est la seule bleue")
plt.show()

---

## Partie D : Un euro demain (10 min, u04)

**Le problème.** On vous propose 9 500 € aujourd'hui, ou 10 000 € **certains** dans un an. Le taux
sans risque vaut $r = 3{,}75\,\%$ en composition continue. Il n'y a aucune incertitude dans cet
énoncé : la réponse est un calcul, pas un avis. Et le même calcul, répété sur trois flux, price une
obligation, donc répond à la question « quand les taux montent, que font les prix obligataires ? »
posée en ouverture du chapitre.

Deux fonctions :

| Fonction | Ce qu'elle rend |
|---|---|
| `actualiser(flux, r, T)` | $V_0 = F_T\,e^{-rT}$, la valeur d'aujourd'hui d'un flux **certain** daté |
| `prix_obligation(coupon, nominal, r, n)` | la somme des $n$ flux actualisés : `coupon * nominal` chaque année, plus le nominal à la fin |

In [ ]:
def actualiser(flux: float, r: float, T: float) -> float:
    """Present value of a single **certain** cash flow paid at date T.

    Parameters
    ----------
    flux : float
        The amount paid at date ``T``, with no uncertainty attached to it.
    r : float
        Continuously compounded annual risk-free rate, in decimal (0.0375 here).
    T : float
        Date of the payment, **in years** (1.5 for eighteen months, never 18).

    Returns
    -------
    float
        ``flux * exp(-r * T)``.
    """
    # TODO: Renvoyer le flux multiplie par le facteur d'actualisation exp(-r*T).
    raise NotImplementedError("TODO")


def prix_obligation(coupon: float, nominal: float, r: float, n: int) -> float:
    """Price of an n-year bond paying a yearly coupon, then the face value back.

    Parameters
    ----------
    coupon : float
        Annual coupon **rate**, in decimal: 0.03 pays ``0.03 * nominal`` every year.
    nominal : float
        Face value, repaid on top of the last coupon.
    r : float
        Continuously compounded annual risk-free rate, in decimal.
    n : int
        Number of yearly payments, the last one carrying the face value.

    Returns
    -------
    float
        Sum of the ``n`` flows, each discounted with its own factor ``exp(-r*k)``.
    """
    dates = np.arange(1, n + 1, dtype=float)
    flux = np.full(n, coupon * nominal)
    flux[-1] += nominal
    # TODO: Renvoyer la somme des flux actualises date par date, de facon vectorisee
    #      avec np.exp(-r * dates) et sans boucle Python.
    raise NotImplementedError("TODO")

In [ ]:
"""The 9 500 today / 10 000 in a year arbitration, then the bond and its rate move."""
valeur_b = actualiser(10_000.0, r, T)
print(f"10 000 EUR certains dans un an valent {valeur_b:,.2f} EUR aujourd'hui "
      f"-> l'offre B bat l'offre A de {valeur_b - 9_500:,.2f} EUR")
print(f"facteur d'actualisation a 10 ans : D(0,10) = {np.exp(-r * 10):.6f}")

prix_375 = prix_obligation(0.03, 100.0, r, 3)
prix_475 = prix_obligation(0.03, 100.0, 0.0475, 3)
print(f"titre 3 ans, coupon 3 % : {prix_375:.2f} a r = 3,75 %, puis {prix_475:.2f} a r = 4,75 %")
print(f"variation : {prix_475 - prix_375:+.2f} points, soit {100 * (prix_475 / prix_375 - 1):+.2f} % "
      "pour +100 points de base de taux")

verifier(
    abs(valeur_b - 9631.94) < 0.01,
    f"OK : {valeur_b:.2f} EUR. Le montant qui rend indifferent entre les deux offres est "
    "celui-la, et il ne depend d'aucune hypothese sur l'avenir.",
    "attendu 9 631,94 EUR : le facteur est exp(-r*T) et non exp(r*T) ; si vous obtenez "
    "10 382, vous avez capitalise au lieu d'actualiser.",
)
verifier(
    abs(prix_375 - 97.71) < 0.01 and abs(prix_475 - 94.91) < 0.01,
    f"OK : {prix_375:.2f} puis {prix_475:.2f}. Quand les taux montent, le prix des obligations "
    "BAISSE : c'est une consequence arithmetique de exp(-rT), rien n'a bouge d'autre.",
    "attendu 97,71 puis 94,91 : le dernier flux vaut coupon*nominal + nominal (103, pas 100), "
    "et chaque flux porte son propre facteur exp(-r*k) pour k = 1, 2, 3.",
)
verifier(
    prix_obligation(0.03, 100.0, 0.0275, 3) > 100.0 > prix_375,
    f"OK : {prix_obligation(0.03, 100.0, 0.0275, 3):.2f} a 2,75 %, au-dessus du pair, contre "
    f"{prix_375:.2f} a 3,75 %. Le prix traverse 100 quand le taux croise le coupon.",
    "attendu un prix superieur a 100 quand r = 2,75 % : verifiez le signe dans exp(-r * dates).",
)

In [ ]:
"""Figure D -- the bond price as a function of the rate, the curve to predict."""
# Cellule FOURNIE : lisez la figure, pas le code.
taux = np.linspace(0.01, 0.07, 200)
prix = np.array([prix_obligation(0.03, 100.0, taux_i, 3) for taux_i in taux])

fig, ax = plt.subplots(figsize=(7.6, 4.4))
ax.plot(100 * taux, prix, lw=1.6, color="#1f4e79")
ax.axhline(100.0, color="grey", ls=":", lw=1.2)
for taux_i, couleur in ((0.0275, "#2e7d32"), (0.0375, "black"), (0.0475, "#c00000")):
    prix_i = prix_obligation(0.03, 100.0, taux_i, 3)
    ax.plot(100 * taux_i, prix_i, "o", color=couleur)
    ax.annotate(f"{100 * taux_i:.2f} % -> {prix_i:.2f}", xy=(100 * taux_i, prix_i),
                xytext=(100 * taux_i + 0.15, prix_i + 1.2), fontsize=9, color=couleur)
ax.set_xlabel("taux sans risque continu r (%)")
ax.set_ylabel("prix du titre (nominal 100)")
ax.set_title("Trois flux, une courbe decroissante : +100 pb de taux, -2,87 % de prix")
plt.show()

---

## Partie E : Payoffs, profit, parité (12 min, u05)

**Le problème.** Vous détenez le portefeuille du cours et vous voulez garder la hausse du S&P 500
sans perdre plus de 10 % sur un an. Avant de choisir un produit, il faut savoir **dessiner** ce
qu'on veut recevoir, puis contrôler ce qu'on lit : un tableau de payoff se vérifie par une somme,
un point mort se calcule avec la prime **capitalisée**, et deux prix affichés se contrôlent l'un par
l'autre sans le moindre modèle.

Données : $S_0 = K = 7\,718{,}60$, $T = 1$, $r = 3{,}75\,\%$, call affiché **546,28**, put affiché
**262,19**. Ces primes sont **posées** : le chapitre 3 dira d'où elles viennent.

Vous écrivez les deux payoffs à la main, puis vous les confrontez aux fonctions fournies.

In [ ]:
def mon_payoff_call(S: np.ndarray, K: float) -> np.ndarray:
    """Payoff of a long call at maturity: max(S - K, 0), element by element.

    Parameters
    ----------
    S : ndarray
        Prices of the underlying at maturity.
    K : float
        Strike.

    Returns
    -------
    ndarray
        What the contract pays, **before** subtracting the premium.
    """
    # TODO: Renvoyer np.maximum applique a S - K et 0.0, sans if et sans boucle.
    raise NotImplementedError("TODO")


def mon_payoff_put(S: np.ndarray, K: float) -> np.ndarray:
    """Payoff of a long put at maturity: max(K - S, 0), element by element.

    Parameters
    ----------
    S : ndarray
        Prices of the underlying at maturity.
    K : float
        Strike.

    Returns
    -------
    ndarray
        What the contract pays, **before** subtracting the premium.
    """
    # TODO: Renvoyer np.maximum applique a K - S et 0.0 : les deux arguments sont
    #      echanges par rapport au call.
    raise NotImplementedError("TODO")

In [ ]:
"""Eight cells, then the column check, then profit -- three scenarios."""
scenarios = np.array([6800.0, 7718.60, 8500.0])
positions = {
    "call long": mon_payoff_call(scenarios, K),
    "call court": -mon_payoff_call(scenarios, K),
    "put long": mon_payoff_put(scenarios, K),
    "put court": -mon_payoff_put(scenarios, K),
}

# The "+ 0.0" turns the negative zeros of the short positions into plain zeros.
print("payoffs        " + "".join(f"{s:>12,.2f}" for s in scenarios))
for nom, valeurs in positions.items():
    print(f"{nom:<14s} " + "".join(f"{v + 0.0:>12,.2f}" for v in valeurs))
somme = sum(positions.values())
print("somme          " + "".join(f"{v + 0.0:>12,.2f}" for v in somme))

verifier(
    np.allclose(positions["call long"], payoff_call(scenarios, K))
    and np.allclose(positions["put long"], payoff_put(scenarios, K)),
    "OK : vos deux payoffs coincident avec les fonctions fournies payoff_call et payoff_put.",
    "attendu l'egalite avec payoff_call / payoff_put : le call est max(S - K, 0), le put "
    "max(K - S, 0) ; c'est l'ordre des deux termes qui les separe.",
)
verifier(
    float(np.abs(somme).max()) < 1e-10,
    "OK : la somme des quatre payoffs est nulle dans chaque colonne. Ce qu'un acheteur "
    "recoit, un vendeur le verse : le contrat ne cree aucune richesse.",
    "attendu une somme nulle colonne par colonne : les positions courtes sont l'OPPOSE des "
    "longues, donc call long + call court doit deja s'annuler (u05 section 4).",
)

In [ ]:
"""Break-even, parity and bounds -- all three without any model of the future."""
prime_capitalisee = PRIME_CALL * np.exp(r * T)
point_mort = K + prime_capitalisee
point_mort_naif = K + PRIME_CALL
profit_call = mon_payoff_call(scenarios, K) - prime_capitalisee

print(f"prime capitalisee : {PRIME_CALL:.2f} x exp({r} x {T}) = {prime_capitalisee:.2f}")
print(f"point mort        : {point_mort:,.2f}  ({100 * (point_mort / S0 - 1):.2f} % au-dessus de S0)")
print(f"calcul spontane   : {point_mort_naif:,.2f}, soit {point_mort - point_mort_naif:.2f} points trop bas")
print("profit du call long " + "".join(f"{v:>12,.2f}" for v in profit_call))

verifier(
    abs(point_mort - 8285.75) < 0.05,
    f"OK : {point_mort:,.2f}. La prime est payee AUJOURD'HUI et le payoff recu dans un an : "
    f"on les ramene a la meme date, d'ou {point_mort - point_mort_naif:.2f} points d'ecart "
    "avec le calcul spontane.",
    "attendu 8 285,75 : le point mort est K + prime CAPITALISEE, soit K + 546,2794*exp(r*T) "
    "et non K + 546,2794 (NOTATION.md section 0.5, et u04 pour la capitalisation).",
)
verifier(
    abs((PRIME_CALL - PRIME_PUT) - (S0 - K * np.exp(-r * T))) < 1e-3,
    f"OK : C - P = {PRIME_CALL - PRIME_PUT:.4f} et S0 - K exp(-rT) = "
    f"{S0 - K * np.exp(-r * T):.4f}. Deux ensembles de flux identiques dans tous les "
    "scenarios ne peuvent pas avoir deux prix : c'est la parite call-put.",
    "attendu l'egalite a 1e-3 pres : le membre de droite actualise K (exp(-rT)), il ne le "
    "capitalise pas ; S0 y entre tel quel.",
)
verifier(
    max(S0 - K * np.exp(-r * T), 0.0) <= PRIME_CALL <= S0,
    f"OK : {max(S0 - K * np.exp(-r * T), 0.0):.2f} <= {PRIME_CALL:.2f} <= {S0:.2f}. Ces bornes "
    "ne supposent rien sur l'avenir : un droit d'acheter ne vaut pas plus que ce qu'il achete.",
    "attendu max(S0 - K exp(-rT), 0) <= C <= S0 : si la borne basse depasse la prime, c'est "
    "que le facteur d'actualisation est applique du mauvais cote.",
)

# The premium never enters the exercise decision: at 7 900 one exercises and still loses.
paye = float(mon_payoff_call(np.array([7900.0]), K)[0])
print(f"\na S_T = 7 900 : le contrat verse {paye:.2f}, l'operation se solde par "
      f"{paye - prime_capitalisee:+.2f} -- et on exerce quand meme, "
      "car a l'echeance le choix est entre 181,40 et 0.")

In [ ]:
"""Figure E -- payoff and profit, same vertical scale (this is the whole point)."""
# Cellule FOURNIE : lisez la figure, pas le code.
prime_capitalisee = PRIME_CALL * np.exp(r * T)
point_mort = K + prime_capitalisee
grille = np.linspace(6000.0, 9500.0, 400)
payoff = mon_payoff_call(grille, K)
profit = payoff - prime_capitalisee

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.4, 4.4), sharey=True)
ax1.plot(grille, payoff, lw=1.8, color="#1f4e79")
ax1.axhline(0.0, color="grey", lw=0.8)
ax1.axvline(K, color="grey", ls=":", lw=1.2)
ax1.set_title(f"(a) Payoff : coude en K = {K:.2f}, jamais negatif")
ax1.set_xlabel("S_T, niveau du SP500 dans un an")
ax1.set_ylabel("euros par unite d'indice")

ax2.plot(grille, profit, lw=1.8, color="#c00000")
ax2.axhline(0.0, color="grey", lw=0.8)
ax2.axhline(-prime_capitalisee, color="#c00000", ls=":", lw=1.0)
ax2.plot([point_mort], [0.0], "o", color="black")
ax2.annotate(f"point mort {point_mort:.2f}", xy=(point_mort, 0.0), xytext=(6100, 620),
             fontsize=9, arrowprops=dict(arrowstyle="->", lw=1.0))
ax2.text(6100, -prime_capitalisee + 60, f"perte maximale {-prime_capitalisee:.2f}",
         fontsize=9, color="#c00000")
ax2.set_title("(b) Profit : la meme courbe, descendue de la prime capitalisee")
ax2.set_xlabel("S_T, niveau du SP500 dans un an")
plt.show()

---

## Clôture : vos trois prédictions, relues (5 min)

Reprenez la partie 0. La cellule ci-dessous met côte à côte ce que vous aviez écrit **avant** et ce
que les 4 151 séances disent. L'écart n'est pas une faute : c'est le contenu du chapitre. Les deux
prédictions que presque tout le monde manque sont la deuxième (on moyenne les volatilités, alors
qu'elles ne s'additionnent pas) et la troisième (on oublie que la prime a été payée un an plus tôt).

Une chose reste inexpliquée à la fin de ce TP : **d'où vient 546,28 ?** Aucun des cinq objets
mesurés ici ne la produit : ni le prix, ni la volatilité, ni la corrélation, ni le taux. Il faudra
savoir mesurer ce qui n'est pas encore arrivé. C'est le chapitre 2.

In [ ]:
"""Confrontation: what you predicted, what the data says, and the gap."""
mesures = {
    "capital_2010_euros": 1000.0 * PRIX["SP500"][-1] / PRIX["SP500"][0],
    "vol_portefeuille_pct": 100.0 * vol_portefeuille(RLOG, POIDS),
    "point_mort_call": K + PRIME_CALL * np.exp(r * T),
}
commentaires = {
    "capital_2010_euros": "indice de prix ; avec les dividendes reinvestis (SPY) : "
                          f"{1000.0 * PRIX['SPY'][-1] / PRIX['SPY'][0]:,.2f} EUR",
    "vol_portefeuille_pct": "moyenne ponderee des six lignes : 19,65 % ; l'ecart est la "
                            "diversification",
    "point_mort_call": "calcul spontane K + prime : 8 264,88 ; il oublie la capitalisation",
}
predictions = globals().get("PREDICTIONS", {})

print(f"{'grandeur':<22s} {'predit':>12s} {'mesure':>12s}   commentaire")
for cle, mesure in mesures.items():
    predit = predictions.get(cle)
    texte = f"{predit:>12,.2f}" if predit is not None else f"{'(a completer)':>12s}"
    print(f"{cle:<22s} {texte} {mesure:>12,.2f}   {commentaires[cle]}")

print(f"\nrappel de vos predictions brutes : {rappel('PREDICTIONS')}")
print("Fin du TP du chapitre 1. Aucun nombre de ce notebook ne vient d'un tirage au hasard.")